# manitex on Colab: train, sample and weave

This notebook runs the whole project on one GPU. It checks the GPU, measures how fast it trains, trains the small model on the procedural grammar of Manipuri textiles, shows the designs the model makes, turns one design into weaving instructions, and exports the model.

**Before you start**

1. *Runtime ▸ Change runtime type ▸* choose the largest GPU you have (**G4**, A100 or H100: 80–96 GB). A T4 or L4 is enough for Sections 1–3 and for a dry run (`SMOKE_TEST = True` below), not for the full training configs.
2. Run the cells from top to bottom. Each section says what to look at.

| Section | What it does | Time |
|---|---|---|
| 1 | Install the code | 3–5 min |
| 2 | Check the GPU, measure the training speed | about 10 min |
| 3 | Sanity run of the training code | 1–5 min |
| 4 | Train the small model (32 M parameters) | 4–9 h on an A100; Section 2 measures your GPU |
| 5 | Training curves and sample sheets | 1 min |
| 6 | Generate designs and weaving instructions | 2 min |
| 7 | Export the model, try the demo, publish | 5–10 min |
| 8 | Optional: base model and baseline | 1–2.5 days and 20–45 h on an A100 |
| 9 | Summary to copy back to Claude | 1 min |

**If the runtime disconnects during Section 4:** reconnect, run Section 1 again, then the Section 4 cell. Training continues from the last checkpoint, which is kept on Google Drive. (If you got a different GPU, run Section 2 again first.)

**What the model learns here.** It is trained on the *procedural grammar* of phanek and Moirang phee, not on photographs of real textiles. Real textiles need the data-collection protocol in `docs/DATA_COLLECTION.md`, and the community's consent.

## 1. Setup

Clones the code, installs it and connects Google Drive, where checkpoints and results are kept so that nothing is lost if the runtime disconnects.

In [ ]:
BRANCH = "main"        # code branch. Until pull request #1 is merged on GitHub, use "ccr-175003d1-cwavnf"
USE_DRIVE = True       # keep checkpoints and results on Google Drive, so the long run survives a disconnect
COMPILE = True         # torch.compile speeds up training; set False if a cell fails with a compile / Triton error
SMOKE_TEST = False     # True: tiny settings that only check that this notebook works (minutes, any runtime)

In [ ]:
import json, os, pathlib

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:                          # not on Colab: a VM, or Jupyter on your own machine
    IN_COLAB = False

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "No GPU (Runtime > Change runtime type)"

if USE_DRIVE and IN_COLAB:
    drive.mount("/content/drive")
    OUT = pathlib.Path("/content/drive/MyDrive/manitex")
else:
    OUT = pathlib.Path(os.environ.get("MANITEX_OUT", "manitex_outputs")).resolve()
OUT.mkdir(parents=True, exist_ok=True)
RUN = OUT / "runs" / "pretrain_small"          # checkpoints of the long run (Section 4)
NUM_WORKERS = min(16, os.cpu_count() or 4)     # CPU processes that make the training designs on the fly
COMPILE_ARG = "compile=" + ("true" if COMPILE and not SMOKE_TEST else "false")


def batch_args():
    # the batch settings found by the speed test of Section 2 (kept in OUT), or '' for the config defaults
    f = OUT / "benchmark.json"
    s = json.load(open(f)).get("suggestion") if f.exists() else None
    return f"batch_size={s['batch_size']} grad_accum={s['grad_accum']}" if s else ""


print("results and checkpoints go to", OUT)

In [ ]:
repo = pathlib.Path("manipuri-textile")
if not pathlib.Path("src/manitex").exists():           # skip when this already is the project folder
    if not repo.exists():
        !git clone -q --depth 1 --branch {BRANCH} https://github.com/chingkheinganba231005/manipuri-textile
    %cd manipuri-textile
assert pathlib.Path("scripts/benchmark_gpu.py").exists(), (
    f"branch {BRANCH!r} does not contain the full project yet: set BRANCH = 'ccr-175003d1-cwavnf' in the settings "
    "cell (or merge pull request #1 on GitHub) and run this cell again")

!python -c "import manitex" 2>/dev/null || pip install -q -e ".[train,hub,figures]"
!python -c "import torch, manitex; print('manitex', manitex.__version__, '| torch', torch.__version__, '| CUDA', torch.version.cuda)"

## 2. Check the GPU and measure the speed (about 10 minutes)

This checks that the installed PyTorch has kernels for your GPU (the G4's RTX PRO 6000 is a Blackwell GPU and needs CUDA 12.8 or newer), then runs real training steps of the small model and reports seconds per step, memory use and how long the full run will take. `--probe` also tries larger batches per step and suggests the largest that fits in memory.

In [ ]:
if SMOKE_TEST:      # a few CPU steps of a tiny model
    bench = "--config configs/train/debug_cpu.yaml --device cpu --steps 2 --warmup 1 crop=8 batch_size=2 grad_accum=2 num_workers=0"
else:
    bench = f"--config configs/train/pretrain_small.yaml --steps 30 --probe {COMPILE_ARG} num_workers={NUM_WORKERS}"
!python scripts/benchmark_gpu.py --json {OUT}/benchmark.json {bench}

In [ ]:
rep = json.load(open(OUT / "benchmark.json"))
env, speed = rep["environment"], rep["benchmark"]
if env["cuda_available"] and not env["kernels_for_gpu"]:
    raise RuntimeError("This PyTorch build has no kernels for your GPU. Run  !pip install -U torch --index-url "
                       "https://download.pytorch.org/whl/cu128  then Runtime > Restart session and start again from Section 1.")
print(f"GPU: {env.get('gpu', 'none (CPU)')} | PyTorch {env['torch']} (CUDA {env['cuda_build']})")
print(f"{speed['sec_per_optimizer_step']:.2f} s per training step, {speed['samples_per_s']:.0f} designs/s, "
      f"peak memory {speed.get('peak_memory_gb', 0):.1f} GB")
print(f"=> the {speed['config_steps']}-step run of Section 4 takes about {speed['projected_hours']:.1f} hours "
      "(plus time for evaluation and sample sheets)")
print("batch settings for the long run:", batch_args() or "the config defaults")

## 3. Sanity run (1–5 minutes)

Trains a tiny model for 500 steps. **What to look for:** the held-out bits per cell (how many bits the model needs per design cell on designs it has not seen; lower is better) must fall from one evaluation to the next, and the sample sheet (one row per style) should show shapes that become cleaner: stripes, temple rows, plaids.

In [ ]:
if SMOKE_TEST:
    sanity = "precision=fp32 num_workers=0 steps=20 eval_every=10 sample_every=20 save_every=20 eval_size=16"
else:
    sanity = f"precision=bf16 num_workers={NUM_WORKERS}"
!manitex train --config configs/train/debug_cpu.yaml out_dir={OUT}/runs/debug {sanity}

In [ ]:
import glob
from IPython.display import Image, display


def show_run(run_dir):
    # held-out score history of a run and its newest sample sheet
    rows = [json.loads(line) for line in open(f"{run_dir}/log.jsonl")]
    evals = [(r["step"], r["eval_bits_per_cell_ema"]) for r in rows if "eval_bits_per_cell_ema" in r]
    print("held-out bits per cell by step:", ", ".join(f"{s}: {b:.3f}" for s, b in evals))
    sheets = sorted(glob.glob(f"{run_dir}/samples/*.png"))
    if sheets:
        print(sheets[-1])
        display(Image(filename=sheets[-1]))


show_run(OUT / "runs" / "debug")

## 4. Train the small model (hours)

32 M parameters, trained on the procedural grammar of five styles (phanek, Moirang phee, banded motif, check, motif tile): 50 000 steps of 256 designs. Section 2 printed the time for your GPU. Keep the browser tab open.

* Checkpoints (`last.pt`, every 5 000 steps) and the log are written to Google Drive. **After a disconnect, run Section 1 again and then this cell: training resumes where it stopped.**
* Out of memory: lower `batch_size` and raise `grad_accum` in the command (the product stays 256).
* A `torch.compile` / Triton error: set `COMPILE = False` in the settings cell, run Section 1's second cell again, then this one.

In [ ]:
if SMOKE_TEST:      # a model of about 1 M parameters for 30 steps
    long_run = ("model=tiny crop=16 batch_size=4 grad_accum=1 steps=30 eval_every=15 sample_every=30 save_every=15 "
                "eval_size=16 precision=fp32 num_workers=0 compile=false")
else:
    long_run = f"{batch_args()} {COMPILE_ARG} num_workers={NUM_WORKERS} log_every=250"
!manitex train --config configs/train/pretrain_small.yaml out_dir={RUN} {long_run}

## 5. Training curves and sample sheets

**What to look for:** the loss and the held-out bits per cell fall and then flatten; accuracy on masked cells rises (it stays below 1 because heavily masked designs are genuinely ambiguous). In the newest sample sheet, each row is one style: phanek should show clean stripes, Moirang phee framed temple rows, checks exact plaids. The sheet is written every 5 000 steps, so you can run this cell while training is still going in another session.

In [ ]:
import matplotlib.pyplot as plt

rows = [json.loads(line) for line in open(RUN / "log.jsonl")]
train = [r for r in rows if "loss" in r]
evals = [r for r in rows if "eval_bits_per_cell_ema" in r]
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
ax[0].plot([r["step"] for r in train], [r["loss"] for r in train])
ax[0].set_title("training loss")
ax[1].plot([r["step"] for r in evals], [r["eval_bits_per_cell_ema"] for r in evals], "o-")
ax[1].set_title("held-out bits per cell (lower is better)")
ax[2].plot([r["step"] for r in train], [r["masked_acc"] for r in train])
ax[2].set_title("accuracy on masked cells")
for a in ax:
    a.set_xlabel("step")
plt.tight_layout()
plt.show()
show_run(RUN)

## 6. Generate designs and weaving instructions

Generates Moirang phee designs that are mirror-symmetric and repeat every 16 cells (the constraints hold *exactly*, not approximately), and compiles each into a weaving draft: a WIF file for weaving software and computer looms, a printable booklet with yarn quantities and pick-up instructions, and a simulated cloth.

All files are in `generated/` on Drive. Open the `.html` booklet in a browser.

In [ ]:
CKPT = RUN / "final.pt" if (RUN / "final.pt").exists() else RUN / "last.pt"      # last.pt while training still runs
GEN = OUT / "generated"
gen = "--size 32x48 --n 2 --steps 8" if SMOKE_TEST else "--size 64x96 --n 4"
print("using", CKPT)
!manitex generate --checkpoint {CKPT} --style moirang_phee {gen} --mirror-x --period-x 16 --seed 0 --render --booklet --out {GEN}

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

previews = sorted(GEN.glob("moirang_phee_0_*_preview.png"))
fig, axes = plt.subplots(1, len(previews), figsize=(4 * len(previews), 4), squeeze=False)
for ax, f in zip(axes[0], previews):
    ax.imshow(plt.imread(f))
    ax.set_title(f.stem.replace("_preview", ""))
    ax.axis("off")
plt.show()
display(Image(filename=str(GEN / "moirang_phee_0_000_cloth.png")))       # simulated cloth of the first design
print("files of the first design:", *sorted(p.name for p in GEN.glob("moirang_phee_0_000*")))
print((GEN / "moirang_phee_0_000.md").read_text()[:1800])               # its weaving instructions

## 7. Export the model, try the demo, publish

* `manitex export` writes a small fp16 `safetensors` model that anyone can load with `TextileGenerator.from_pretrained`.
* The demo is the Gradio app that also runs on Hugging Face Spaces. It gives a temporary public link while the cell runs; stop the cell to end it.
* **Publishing makes the model and the Space public.** This model is trained on procedural data only (no community textiles). Before publishing anything trained on real community data, read `docs/ETHICS.md`.
* To publish you need a free Hugging Face account and a token with *write* access (huggingface.co/settings/tokens). Add it in Colab under *Secrets* (the key icon on the left) as `HF_TOKEN` and switch on *Notebook access*. After the upload, open the Space's *Settings ▸ Hardware* and choose **ZeroGPU** (free for up to two Spaces on a personal account).

In [ ]:
EXPORT = OUT / "exports" / "manitex-small"
!manitex export {CKPT} --out {EXPORT}
!ls -lh {EXPORT}

In [ ]:
if SMOKE_TEST:
    print("dry run: skipping the demo")
else:
    !pip install -q "gradio>=4.44"
    !manitex app --checkpoint {EXPORT} --share

In [ ]:
HF_USER = ""          # your Hugging Face user name; leave it empty to skip publishing
if HF_USER and not SMOKE_TEST:
    from google.colab import userdata
    from huggingface_hub import HfApi

    api = HfApi(token=userdata.get("HF_TOKEN"))                       # the write token stored under Colab Secrets
    api.create_repo(f"{HF_USER}/manitex-small", repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=str(EXPORT), repo_id=f"{HF_USER}/manitex-small")
    !python scripts/build_space.py --out space_build --model {EXPORT} --hardware zerogpu
    api.create_repo(f"{HF_USER}/manitex", repo_type="space", space_sdk="gradio", exist_ok=True)
    api.upload_folder(folder_path="space_build", repo_id=f"{HF_USER}/manitex", repo_type="space")
    print(f"model: https://huggingface.co/{HF_USER}/manitex-small")
    print(f"demo:  https://huggingface.co/spaces/{HF_USER}/manitex   (now choose ZeroGPU in its Settings > Hardware)")

## 8. Optional: the larger runs

Switch a flag on to start a run. Both resume from Drive after a disconnect, so a run that needs more than one Colab session simply continues when you run the cell again. The numbers are A100 estimates; the table in `docs/TRAINING.md` explains how they were counted.

* `RUN_BASE`: the main model (129 M parameters, `configs/train/pretrain_base.yaml`): 1–2.5 days. Train the small model first; if its held-out score flattens early, shorten the run with `steps=...`.
* `RUN_AR`: the autoregressive baseline of experiment E1 (`configs/train/baseline_ar_small.yaml`): 20–45 hours.

Fine-tuning on real data, the other experiments (E1–E7) and the weaver study need the real dataset first; see `docs/RESEARCH_PLAN.md` and `docs/DATA_COLLECTION.md`.

In [ ]:
RUN_BASE = False
RUN_AR = False
if RUN_BASE:
    !manitex train --config configs/train/pretrain_base.yaml out_dir={OUT}/runs/pretrain_base {COMPILE_ARG} num_workers={NUM_WORKERS} log_every=250
if RUN_AR:
    !manitex train --config configs/train/baseline_ar_small.yaml out_dir={OUT}/runs/baseline_ar_small {COMPILE_ARG} num_workers={NUM_WORKERS} log_every=250

## 9. Summary to copy back

Copy the output of this cell into the chat with Claude. It holds the GPU, the measured speed, the suggested batch settings and the state of the training run, which is what the time estimates, batch sizes and the paper's numbers are based on.

When you are done, use *Runtime ▸ Disconnect and delete runtime* so the GPU stops using your compute units.

In [ ]:
def summary():
    out = {}
    f = OUT / "benchmark.json"
    if f.exists():
        r = json.load(open(f))
        out["environment"] = {k: r["environment"].get(k) for k in
                              ("gpu", "memory_gb", "capability", "driver", "torch", "cuda_build", "kernels_for_gpu")}
        out["bf16_matmul_tflops"] = r.get("matmul_tflops_bf16")
        out["speed"] = {k: r["benchmark"].get(k) for k in
                        ("sec_per_optimizer_step", "samples_per_s", "peak_memory_gb", "projected_hours")}
        out["suggested_batch"] = r.get("suggestion")
        out["probe"] = r.get("probe")
    log = RUN / "log.jsonl"
    if log.exists():
        rows = [json.loads(line) for line in open(log)]
        evals = [r for r in rows if "eval_bits_per_cell_ema" in r]
        timed = [r for r in rows if "samples_per_s" in r]
        out["training"] = {"last_step": rows[-1]["step"], "last_eval": evals[-1] if evals else None,
                           "samples_per_s": timed[-1]["samples_per_s"] if timed else None,
                           "gpu_mem_gb": timed[-1].get("gpu_mem_gb") if timed else None}
    return out


print(json.dumps(summary(), indent=1))